# Success Rate (SR) Test
SR = progress toward solving the puzzle after `max_steps` steps.
- Solved → 100%
- Not solved → correct_positions / total_positions

Demonstrates `compute_progress()` on both SlidingPuzzle and TowerOfHanoi without LLM/GPU.

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("")), "src"))

from sliding_puzzle.enviroment import SlidingPuzzle
from tower_of_hanoi.enviroment import TowerOfHanoi

print("Setup OK")

Setup OK


## Test 1: Solved SlidingPuzzle → SR = 100%

In [2]:
# Goal state is now [0,1,2,3,4,5,6,7,8]
# Use CONFIGS for a known solvable state
from sliding_puzzle.enviroment import CONFIGS

# 2x2 puzzle: [2,1,3,0] → goal [0,1,2,3]
puzzle = SlidingPuzzle(initial_state=CONFIGS["2x2"])
print(f"State: {puzzle.get_state()}, Goal: {puzzle.goal_state}")
sr_initial = puzzle.compute_progress()
print(f"Initial SR: {sr_initial:.1%}")

# Manually check: state=[2,1,3,0], goal=[0,1,2,3]
# idx0: 2≠0, idx1: 1=1 ✓, idx2: 3≠2, idx3: 0≠3 → 1/4 = 25%
assert sr_initial == 1/4, f"Expected 25%, got {sr_initial:.1%}"
print("PASS: initial state SR=25%")

State: [2, 1, 3, 0], Goal: [0, 1, 2, 3]
Initial SR: 25.0%
PASS: initial state SR=25%


## Test 2: Partially solved SlidingPuzzle → SR = correct/total

In [3]:
# Goal is now [0,1,2,3,4,5,6,7,8]
# State [0,1,2,5,6,3,4,7,8]:
#   idx0: 0=0 ✓  idx1: 1=1 ✓  idx2: 2=2 ✓
#   idx3: 5≠3    idx4: 6≠4    idx5: 3≠5
#   idx6: 4≠6    idx7: 7=7 ✓  idx8: 8=8 ✓
# → 5/9 correct

puzzle = SlidingPuzzle(initial_state=[0, 1, 2, 5, 6, 3, 4, 7, 8])
sr = puzzle.compute_progress()
print(f"State:    {puzzle.get_state()}")
print(f"Goal:     {puzzle.goal_state}")
print(f"SR: {sr:.1%} (5/9 tiles correct)")
assert sr == 5/9, f"Expected 55.6%, got {sr:.1%}"
print("PASS: partial progress → SR=55.6%")

# State with fewer correct tiles
# [1, 2, 5, 6, 3, 4, 7, 8, 0] → goal [0,1,2,3,4,5,6,7,8]
# idx0: 1≠0, idx1: 2≠1, ... → check
puzzle2 = SlidingPuzzle(initial_state=CONFIGS["3x3 (easiest)"])
sr2 = puzzle2.compute_progress()
print(f"\nState:    {puzzle2.get_state()}")
print(f"Goal:     {puzzle2.goal_state}")
print(f"SR: {sr2:.1%}")
print("PASS: another partial state computed")

State:    [0, 1, 2, 5, 6, 3, 4, 7, 8]
Goal:     [0, 1, 2, 3, 4, 5, 6, 7, 8]
SR: 55.6% (5/9 tiles correct)
PASS: partial progress → SR=55.6%

State:    [1, 2, 5, 6, 3, 4, 7, 8, 0]
Goal:     [0, 1, 2, 3, 4, 5, 6, 7, 8]
SR: 0.0%
PASS: another partial state computed


## Test 3: SlidingPuzzle progress improves after correct moves

In [4]:
# 2x2 puzzle: [2,1,3,0] → goal [0,1,2,3]
# Move tile 2 up: blank at idx3, tile 2 at idx0
puzzle = SlidingPuzzle(initial_state=[2, 1, 3, 0])
sr_before = puzzle.compute_progress()
print(f"Before: {puzzle.get_state()} → SR={sr_before:.1%}")

# Move tile 3 left (blank is at idx 3, tile 3 is at idx 2)
puzzle.apply_move((3, "right"))
sr_after = puzzle.compute_progress()
print(f"After:  {puzzle.get_state()} → SR={sr_after:.1%}")
print(f"Progress changed: {sr_before:.1%} → {sr_after:.1%}")
print("PASS: SR changes after moves")

Before: [2, 1, 3, 0] → SR=25.0%
After:  [2, 1, 0, 3] → SR=50.0%
Progress changed: 25.0% → 50.0%
PASS: SR changes after moves


## Test 4: TowerOfHanoi — disks on target peg = progress

In [5]:
# 3-disk Tower of Hanoi: all disks start on peg 0, goal is peg 2
hanoi = TowerOfHanoi(num_disks=3)
sr0 = hanoi.compute_progress()
print(f"Initial: {hanoi.get_state()} → SR={sr0:.1%} (0/3 on target)")
assert sr0 == 0.0

# Move disk 1 to peg 2
hanoi.move_disk(1, 0, 2)
sr1 = hanoi.compute_progress()
print(f"After move 1→peg2: {hanoi.get_state()} → SR={sr1:.1%} (1/3 on target)")
assert sr1 == 1 / 3

# Solve it: move disk 1 off, disk 2 to peg 1, disk 1 to peg 1, disk 3 to peg 2, etc.
# Reset and solve properly
hanoi2 = TowerOfHanoi(num_disks=3)
moves = [(1,0,2), (2,0,1), (1,2,1), (3,0,2), (1,1,0), (2,1,2), (1,0,2)]
for m in moves:
    hanoi2.move_disk(*m)
sr_solved = hanoi2.compute_progress()
print(f"Solved: {hanoi2.get_state()} → SR={sr_solved:.1%}")
assert hanoi2.is_solved()
assert sr_solved == 1.0
print("PASS: TowerOfHanoi progress = disks_on_target / total")

Initial: [[3, 2, 1], [], []] → SR=0.0% (0/3 on target)
After move 1→peg2: [[3, 2], [], [1]] → SR=33.3% (1/3 on target)
Solved: [[], [], [3, 2, 1]] → SR=100.0%
PASS: TowerOfHanoi progress = disks_on_target / total


## Summary
All tests passed:
- **Test 1**: Solved SlidingPuzzle → SR=100%
- **Test 2**: Partial progress → SR = correct_tiles / total (0%, 55.6%)
- **Test 3**: SR increases after correct moves
- **Test 4**: TowerOfHanoi — SR = disks_on_target / total (0% → 33.3% → 100%)